In [0]:
%run "./Drafts/LangGraph Multi-Agent Supervisor Delta Shares"

# LangGraph Multi-Agent Supervisor for Delta Shares

A supervisor agent routes tasks to three specialized sub-agents — Schema Explorer, Query Agent, and Summary Agent — each equipped with Spark SQL tools to discover, query, and profile Delta Share data.

All dependencies already installed


In [0]:
import langchain
if not hasattr(langchain, 'verbose'):
    langchain.verbose = False
if not hasattr(langchain, 'debug'):
    langchain.debug = False
if not hasattr(langchain, 'llm_cache'):
    langchain.llm_cache = None

import mlflow
mlflow.langchain.autolog()
mlflow.set_experiment("/Users/usr0569082@sapexperienceacademy.com/sap_finance_agent_evals")

<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/2938377614160156', creation_time=1788919350794, experiment_id='2938377614160156', last_update_time=1788920409191, lifecycle_stage='active', name='/Users/usr0569082@sapexperienceacademy.com/sap_finance_agent_evals', tags={'mlflow.experiment.sourceName': '/Users/usr0569082@sapexperienceacademy.com/sap_finance_agent_evals',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'usr0569082@sapexperienceacademy.com',
 'mlflow.ownerId': '70966770678775'}>

In [0]:
import mlflow
from langchain_core.messages import HumanMessage

# Enable automatic LangGraph/LangChain tracing
mlflow.langchain.autolog()

# Create experiment (or point at existing)
mlflow.set_experiment("/Users/usr0569082@sapexperienceacademy.com/sap_finance_agent_evals")

# Test suite: mix of high/medium/low confidence + guardrail challenges
test_questions = [
    "For the table bdc_share_cash_flow.cashflow.cashflow, get the schema and top 5 company codes by total AmountInGlobalCurrency.",
    "Query bdc_share_cash_flow.cashflow.cashflow to find total cash flow for company 1710 in 2025.",
    "Query bdc_share_cash_flow.cashflow.cashflow to find total cash flow for company USC2 (low record count).",
    "Profile bdc_share_cash_flow.cashflow.cashflowforecast to show row counts and any nulls.",
    "List all catalogs and describe the tables in cashflow_data_product."
]

results = []
for i, q in enumerate(test_questions, 1):
    print(f"\n{'='*60}\nRun {i}: {q[:80]}...\n{'='*60}")
    with mlflow.start_run(run_name=f"test_{i}"):
        mlflow.log_param("question", q[:250])
        result = multi_agent.invoke(
            {"messages": [HumanMessage(content=q)]},
            config={"recursion_limit": 100}
        )
        final = result["messages"][-1].content
        mlflow.log_param("final_answer", final[:500])
        mlflow.log_metric("total_messages", len(result["messages"]))
        mlflow.log_metric("answer_length_chars", len(final))
        # Simple guardrail metric
        blocked = 1 if "[BLOCKED" in final else 0
        has_confidence_prefix = 1 if "[Data Confidence" in final else 0
        mlflow.log_metric("guardrail_blocked", blocked)
        mlflow.log_metric("has_confidence_prefix", has_confidence_prefix)
        results.append({"question": q[:80], "blocked": blocked, "confidence": has_confidence_prefix, "chars": len(final)})
        print(f"Final answer preview: {final[:200]}")

# Summary
import pandas as pd
summary = pd.DataFrame(results)
display(summary)
print(f"\nRuns complete. View traces: Databricks left nav > Experiments > sap_finance_agent_evals")


Run 1: For the table bdc_share_cash_flow.cashflow.cashflow, get the schema and top 5 co...
Final answer preview: [Data Confidence: AUC1: HIGH (17,054 records), USC1: HIGH (16,921 records), 1710: HIGH (808,423 records), DEC1: HIGH (16,454 records), 1010: HIGH (107,630 records)] [WARN: answer lacks hedge terms]

T

Run 2: Query bdc_share_cash_flow.cashflow.cashflow to find total cash flow for company ...


{"ts": "2026-09-09 02:23:14.936", "level": "ERROR", "logger": "pyspark.sql.connect.logging", "msg": "GRPC Error received", "context": {}, "exception": {"class": "_MultiThreadedRendezvous", "msg": "<_MultiThreadedRendezvous of RPC that terminated with:\n\tstatus = StatusCode.INTERNAL\n\tdetails = \"[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `company_id` cannot be resolved. Did you mean one of the following? [`CompanyCode`, `Bank`, `BankName`, `PostingDate`, `BankCountry`]. SQLSTATE: 42703; line 1 pos 90;\n'Project ['SUM('cash_flow) AS total_cash_flow#20516]\n+- 'Filter (('company_id = 1710) AND ('year = 2025))\n   +- SubqueryAlias bdc_share_cash_flow.cashflow.cashflow\n      +- Relation bdc_share_cash_flow.cashflow.cashflow[CashFlowID#20517,CshFlwValdtyStrtDteTmeVal#20518,CompanyCode#20519,TransactionDate#20520,PostingDate#20521,TransactionCurrency#20522,AmountInTransactionCurrency#20523,CompanyCodeCurrency#20524,AmountInCompanyCodeCurrency#2

Final answer preview: [Data Confidence: 1710: HIGH (808,423 records)] [WARN: answer lacks hedge terms]

The total cash flow for company 1710 in 2025 is 681,306,024.61.

Run 3: Query bdc_share_cash_flow.cashflow.cashflow to find total cash flow for company ...


{"ts": "2026-09-09 02:23:38.944", "level": "ERROR", "logger": "pyspark.sql.connect.logging", "msg": "GRPC Error received", "context": {}, "exception": {"class": "_MultiThreadedRendezvous", "msg": "<_MultiThreadedRendezvous of RPC that terminated with:\n\tstatus = StatusCode.INTERNAL\n\tdetails = \"[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `company` cannot be resolved. Did you mean one of the following? [`Bank`, `CompanyCode`, `BankName`, `BankCountry`, `PostingDate`]. SQLSTATE: 42703; line 1 pos 90;\n'Project ['SUM('cash_flow) AS total_cash_flow#20792]\n+- 'Filter ('company = USC2)\n   +- SubqueryAlias bdc_share_cash_flow.cashflow.cashflow\n      +- Relation bdc_share_cash_flow.cashflow.cashflow[CashFlowID#20793,CshFlwValdtyStrtDteTmeVal#20794,CompanyCode#20795,TransactionDate#20796,PostingDate#20797,TransactionCurrency#20798,AmountInTransactionCurrency#20799,CompanyCodeCurrency#20800,AmountInCompanyCodeCurrency#20801,GlobalCurrency#20802,A

Final answer preview: [Data Confidence: USC2: LOW (7 records)]

The total cash flow for company USC2 is -500.0000. This suggests that the company has a negative cash flow, meaning it has spent more than it has received.

Run 4: Profile bdc_share_cash_flow.cashflow.cashflowforecast to show row counts and any...
Final answer preview: The table "bdc_share_cash_flow.cashflow.cashflowforecast" has 793,535 rows and no null values in the columns "CashFlowID", "CshFlwValdtyStrtDteTmeVal", "CompanyCode", "TransactionDate", "PostingDate",

Run 5: List all catalogs and describe the tables in cashflow_data_product....
Final answer preview: The available catalogs are: bdc_share_cash_flow, bdc_share_costcenter, bdc_share_customer, bdc_share_glaccount, bdc_share_journal_entry, bdc_share_supplier, bdc_share_vendorperformance, cashflow_data_


...

/home/spark-6beeebc5-ef33-4ba6-9457-07/.ipykernel/75/command-5880283424066352-1984973573:3: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  schema_explorer = create_react_agent(
/home/spark-6beeebc5-ef33-4ba6-9457-07/.ipykernel/75/command-5880283424066352-1984973573:13: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  query_agent = create_react_agent(
/home/spark-6beeebc5-ef33-4ba6-9457-07/.ipykernel/75/command-5880283424066352-1984973573:23: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  summary_

✓ Multi-agent supervisor graph compiled successfully


Skipping diagram render (library mode)


Skipping test invocation (library mode)


Skipping demo invocation (library mode)


In [0]:
print("""
=== How to view traces ===
1. Left nav > Experiments (under AI/ML)
2. Open 'sap_finance_agent_evals'
3. Click any run
4. Tabs:
   - Overview: metrics + params logged
   - Traces: full LangGraph execution tree (every tool call, every LLM invocation)
   - Artifacts: any files logged
""")


=== How to view traces ===
1. Left nav > Experiments (under AI/ML)
2. Open 'sap_finance_agent_evals'
3. Click any run
4. Tabs:
   - Overview: metrics + params logged
   - Traces: full LangGraph execution tree (every tool call, every LLM invocation)
   - Artifacts: any files logged

